In [0]:
%run /Workspace/Users/molugurikoushik68@gmail.com/banking-agent/notebooks/01_document_processor

In [0]:

"""
Embeddings Generator
====================

Converts policy chunks into numerical vectors using Databricks.

Process:
1. Load chunks from previous step (document_processor)
2. Call Databricks embedding model (gte-large-en)
3. Get 1024-dimensional vectors for each chunk
4. Store vectors with metadata
5. Display sample embeddings

Key concept:
- Text → numbers (embeddings)
- Similar text → similar numbers
- Enables semantic search
"""

import json
import pandas as pd
from typing import List, Dict, Tuple
import requests
from openai import OpenAI

# ============================================
# 1. FUNCTION: Get Embedding from Databricks
# ============================================

def get_embedding(text: str, endpoint_name: str = "gte-large-en") -> List[float]:
    """
    Get embedding vector from Databricks Foundation Model.
    
    What it does:
    1. Call Databricks API with text
    2. Model creates 1024 numbers representing text meaning
    3. Return the vector
    
    Why:
    - Databricks provides free embedding models
    - gte-large-en is specifically trained for retrieval
    - Returns consistent vectors (same text = same vector)
    
    Args:
        text: Text to embed (e.g., policy chunk)
        endpoint_name: Model name (default: gte-large-en)
        
    Returns:
        List of 1024 floats (the embedding)
        
    Example:
        text = "Transfer limit is $10,000"
        vector = get_embedding(text)
        # Returns: [0.123, 0.456, ..., 0.789] (1024 numbers)
    """
    
    # In Databricks, use the built-in embedding model
    # This uses Mosaic AI, which is free for Community Edition
    
    # We'll use a simpler approach: call Databricks Foundation Model via serving endpoint
    # For this demo, we'll use a mock embedding (in production, use actual endpoint)
    
    # Production code would be:
    """
    from databricks.vector_search.client import VectorSearchClient
    client = VectorSearchClient()
    response = client.search_index(
        index_name="policy_embeddings",
        query_text=text,
        num_results=1
    )
    """
    

# Databricks Foundation Models endpoint
client = OpenAI(
    base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints",
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

def get_embedding(
    text: str,
    endpoint_name: str = "databricks-bge-large-en"
) -> List[float]:
    """
    Generate REAL embeddings using Databricks Foundation Models.
    Uses actual semantic embedding model, not random vectors.
    """
    response = client.embeddings.create(
        model=endpoint_name,
        input=text
    )
    return response.data[0].embedding
    


# ============================================
# 2. FUNCTION: Batch Generate Embeddings
# ============================================

def generate_embeddings_batch(chunks: List[str], 
                            batch_size: int = 10) -> List[Dict]:
    """
    Generate embeddings for all chunks.
    
    What it does:
    1. Take list of chunks
    2. For each chunk, call embedding model
    3. Store chunk + embedding together
    4. Return list of {chunk: text, embedding: vector}
    
    Why batch?
    - Processing multiple at once is more efficient
    - Can track progress
    - Handle errors gracefully
    
    Args:
        chunks: List of policy chunks (from document processor)
        batch_size: How many to process at once (default 10)
        
    Returns:
        List of dicts: [
            {
                'chunk_id': 0,
                'text': 'Account policies...',
                'embedding': [0.123, 0.456, ...],
                'dimension': 1024,
                'length': 156 (words)
            },
            ...
        ]
        
    Example:
        chunks = ["Transfer...", "Refund...", ...]
        embeddings = generate_embeddings_batch(chunks)
        # Each chunk now has a 1024-dimensional vector
    """
    
    print(f"Generating embeddings for {len(chunks)} chunks...")
    print(f"Batch size: {batch_size}")
    
    embeddings_data = []
    
    for i, chunk in enumerate(chunks):
        # Show progress
        if (i + 1) % batch_size == 0 or i == len(chunks) - 1:
            print(f"  Processing chunk {i+1}/{len(chunks)}...")
        
        # Get embedding
        vector = get_embedding(chunk)
        
        if not vector:
            print(f"  ✗ Failed to embed chunk {i}")
            continue
        
        # Store with metadata
        embedding_record = {
            'chunk_id': i,
            'text': chunk,
            'embedding': vector,
            'word_count': len(chunk.split()),
            'char_count': len(chunk),
            'embedding_dimension': len(vector),
        }
        
        embeddings_data.append(embedding_record)
    
    print(f"✓ Generated embeddings for {len(embeddings_data)} chunks")
    return embeddings_data

# ============================================
# 3. FUNCTION: Calculate Similarity
# ============================================

def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """
    Calculate cosine similarity between two vectors.
    
    What it does:
    - Compares two embedding vectors
    - Returns value between 0.0 and 1.0
    - 1.0 = identical, 0.0 = completely different
    
    Why:
    - Used to find similar policies
    - Foundation of semantic search
    
    Formula:
    similarity = dot_product(vec1, vec2) / (magnitude1 * magnitude2)
    
    Args:
        vec1: First embedding vector
        vec2: Second embedding vector
        
    Returns:
        Similarity score (0.0 to 1.0)
        
    Example:
        vec1 = [0.1, 0.2, 0.3]
        vec2 = [0.1, 0.2, 0.3]
        similarity = cosine_similarity(vec1, vec2)
        # Returns: 1.0 (identical)
    """
    
    # Dot product
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    
    # Magnitudes (should be 1.0 for normalized vectors, but calculate anyway)
    import math
    mag1 = math.sqrt(sum(x**2 for x in vec1))
    mag2 = math.sqrt(sum(x**2 for x in vec2))
    
    if mag1 == 0 or mag2 == 0:
        return 0.0
    
    return dot_product / (mag1 * mag2)

# ============================================
# 4. FUNCTION: Create Dataframe
# ============================================

def create_embeddings_dataframe(embeddings_data: List[Dict]) -> pd.DataFrame:
    """
    Convert embeddings to Pandas dataframe for storage.
    
    What it does:
    - Takes list of embeddings
    - Converts to Pandas dataframe
    - Creates columns: chunk_id, text, embedding, etc.
    
    Why:
    - Dataframe is easier to work with
    - Can save to Databricks table
    - Can query easily
    
    Args:
        embeddings_data: List of embedding records
        
    Returns:
        Pandas dataframe
    """
    
    df = pd.DataFrame(embeddings_data)
    
    print(f"\nEmbeddings Dataframe:")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Data types:")
    for col, dtype in df.dtypes.items():
        print(f"    - {col}: {dtype}")
    
    return df

# ============================================
# 5. FUNCTION: Display Sample Embeddings
# ============================================

def display_sample_embeddings(embeddings_data: List[Dict], 
                             num_samples: int = 2):
    """
    Show sample embeddings for inspection.
    
    Args:
        embeddings_data: List of embedding records
        num_samples: How many to display
    """
    
    print(f"\nShowing {min(num_samples, len(embeddings_data))} sample embeddings:\n")
    
    for record in embeddings_data[:num_samples]:
        print(f"--- CHUNK {record['chunk_id']} ---")
        print(f"Text: {record['text'][:100]}...")
        print(f"Word count: {record['word_count']}")
        print(f"Embedding dimension: {record['embedding_dimension']}")
        print(f"First 10 values: {record['embedding'][:10]}")
        print(f"Similarity to self: {cosine_similarity(record['embedding'], record['embedding']):.4f}")
        print()

# ============================================
# 6. FUNCTION: Test Similarity Search
# ============================================

def test_similarity_search(embeddings_data: List[Dict], 
                          query_text: str) -> List[Tuple[int, float]]:
    """
    Test: Find similar chunks for a query.
    
    What it does:
    1. Embed the query
    2. Compare to all chunks
    3. Return top similar chunks
    
    Why:
    - Verify embeddings work
    - Show how semantic search works
    
    Args:
        embeddings_data: List of embedding records
        query_text: Query text (e.g., "What is transfer limit?")
        
    Returns:
        List of (chunk_id, similarity_score) tuples
    """
    
    print(f"\nTest Query: '{query_text}'")
    
    # Embed query
    query_vector = get_embedding(query_text)
    
    # Calculate similarity to all chunks
    similarities = []
    for record in embeddings_data:
        chunk_id = record['chunk_id']
        vector = record['embedding']
        similarity = cosine_similarity(query_vector, vector)
        similarities.append((chunk_id, similarity))
    
    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Display top 3
    print(f"\nTop 3 similar chunks:")
    for rank, (chunk_id, similarity) in enumerate(similarities[:3], 1):
        text = embeddings_data[chunk_id]['text']
        print(f"{rank}. Chunk {chunk_id}: {similarity:.4f}")
        print(f"   Text: {text[:80]}...")
    
    return similarities

# ============================================
# 7. MAIN EXECUTION
# ============================================

# Configuration and chunks are already loaded from Cell 1 (%run 01_document_processor)
# Available: CHUNK_SIZE, EMBEDDING_MODEL, EMBEDDING_DIMENSION, processed_chunks

print("="*60)
print("EMBEDDINGS GENERATOR: Converting Chunks to Vectors")
print("="*60)
print(f"Model: {EMBEDDING_MODEL}")
print(f"Dimension: {EMBEDDING_DIMENSION}")
print(f"Input chunks: {len(chunks)}")

# Generate embeddings
embeddings_data = generate_embeddings_batch(chunks, batch_size=5)

if embeddings_data:
    # Create dataframe
    embeddings_df = create_embeddings_dataframe(embeddings_data)
    
    # Display samples
    display_sample_embeddings(embeddings_data, num_samples=2)
    
    # Test semantic search
    test_queries = [
        "What is the transfer limit?",
        "How long does refund take?",
        "What is minimum account balance?"
    ]
    
    for query in test_queries:
        test_similarity_search(embeddings_data, query)
        print()
    
    # Export for next step
    print("="*60)
    print("SUMMARY")
    print("="*60)
    print(f"✓ Generated {len(embeddings_data)} embeddings")
    print(f"✓ Dimension: {EMBEDDING_DIMENSION}")
    print(f"✓ Ready to store in vector database (next step)")
    
    # Store for next step
    embeddings_for_storage = embeddings_data
    embeddings_dataframe = embeddings_df
    
    print(f"\n✓ Embeddings stored in:")
    print(f"  - embeddings_for_storage (list)")
    print(f"  - embeddings_dataframe (Pandas dataframe)")
else:
    print("✗ No embeddings generated. Check previous steps.")
